# Tool Usage Diversity

The Tool Usage Diversity notebook contains the implementation, tests, and usage workflow for the Tool Usage Diversity metric. It scans JSON/JSONL conversation traces, counts structured assistant tool calls, and calculates normalized entropy against one fixed tool inventory.

The score is `H / ln(K)`, where `K` is the number of configured tools. A reference tool with no calls remains in the denominator. No recognized calls means the score is `None` (undefined). Unknown tools are reported and excluded from scoring. A one-tool inventory scores `0.0` when it has calls. The score measures the aggregate tool-call distribution, not equality of usage on every task.

The parser handles structured `tool_calls` and legacy `function_call` fields on assistant/AI messages. It does not infer calls from text or tool results. The supplied tau2 `results.json` layout (`simulations` with `messages`) has been checked; other benchmark output formats may need a parser adjustment.

To run: open this notebook, run the cells from top to bottom, edit the configuration in the final cell, then run that cell. The tests use only Python's standard library. You will need to have ran tau-bench as well as know the paths to the input and output of your results.

In [ ]:
from collections import Counter
from pathlib import Path
from typing import Any
import json
import math
import sys

cwd = Path.cwd()
repo_root = next(
    (path for path in (cwd, *cwd.parents) if (path / "evaluation" / "tool_usage_diversity.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Start this notebook inside the project repository.")
sys.path.insert(0, str(repo_root))

from evaluation.tool_usage_diversity import calculate_diversity, validate_perturbations


def tool_name(call: Any) -> str | None:
    """Get a tool name from a common structured tool-call object."""
    if not isinstance(call, dict):
        return None
    name = call.get("name")
    if not isinstance(name, str) and isinstance(call.get("function"), dict):
        name = call["function"].get("name")
    return name.strip() if isinstance(name, str) and name.strip() else None


def extract_tool_calls(record: Any) -> list[str]:
    """Extract structured calls from one message list or conversation record."""
    if isinstance(record, list):
        messages = record
    elif isinstance(record, dict):
        messages = next(
            (record[key] for key in ("messages", "conversation", "trajectory")
             if isinstance(record.get(key), list)),
            None,
        )
        if messages is None and record.get("role", record.get("type")) in {
            "assistant", "ai", "AIMessage"
        }:
            messages = [record]
    else:
        return []

    if not isinstance(messages, list):
        return []

    calls, seen_ids = [], set()
    for message in messages:
        if not isinstance(message, dict):
            continue
        payload = message.get("data", message)
        if not isinstance(payload, dict):
            continue
        role = str(message.get("role", message.get("type", ""))).lower()
        if role not in {"assistant", "ai", "aimessage"}:
            continue

        raw_calls = payload.get("tool_calls")
        if not isinstance(raw_calls, list):
            extra = payload.get("additional_kwargs", {})
            raw_calls = extra.get("tool_calls") if isinstance(extra, dict) else None
        if not isinstance(raw_calls, list):
            legacy = payload.get("function_call")
            raw_calls = [legacy] if isinstance(legacy, dict) else []

        for call in raw_calls:
            name = tool_name(call)
            if name is None:
                continue
            call_id = call.get("id") if isinstance(call, dict) else None
            if call_id and call_id in seen_ids:
                continue
            if call_id:
                seen_ids.add(call_id)
            calls.append(name)
    return calls

In [ ]:
def read_records(path: Path) -> list[Any]:
    """Read JSON records or JSONL messages/tasks from a file."""
    if path.suffix.lower() == ".json":
        value = json.loads(path.read_text(encoding="utf-8-sig"))
        if isinstance(value, dict):
            simulations = value.get("simulations")
            if isinstance(simulations, list):
                info = value.get("info", {})
                environment = info.get("environment_info", {}) if isinstance(info, dict) else {}
                source_domain = environment.get("domain_name") if isinstance(environment, dict) else None
                return [dict(row, _source_domain=source_domain)
                        for row in simulations if isinstance(row, dict)]
            return value.get("tasks", [value])
        if isinstance(value, list):
            return [value] if value and all(isinstance(row, dict) and "role" in row for row in value) else value
        return []
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8-sig").splitlines()
            if line.strip()]
    if rows and all(isinstance(row, dict) and ("role" in row or "type" in row) for row in rows):
        return [rows]
    return [record for row in rows
            for record in (row.get("tasks", [row]) if isinstance(row, dict) else [row])]


def analyze_dataset(
    input_paths: list[Path],
    reference_tools: list[str],
    task_ids: set[str] | None = None,
    domain: str | None = None,
) -> dict[str, Any]:
    """Scan JSON/JSONL inputs, count calls, and return metrics and diagnostics."""
    files = set()
    for path in input_paths:
        if not path.exists():
            raise FileNotFoundError(f"Input does not exist: {path}")
        if path.is_file():
            if path.suffix.lower() not in {".json", ".jsonl"}:
                raise ValueError(f"Unsupported input file type: {path}")
            files.add(path.resolve())
        else:
            files.update(item.resolve() for item in path.rglob("*")
                         if item.is_file() and item.suffix.lower() in {".json", ".jsonl"})

    counts, task_calls, warnings = Counter(), [], []
    parsed_files = skipped_files = conversation_files = task_records = 0
    if not files:
        warnings.append("No JSON or JSONL input files found.")

    for path in sorted(files, key=lambda item: str(item).casefold()):
        try:
            records = read_records(path)
        except (OSError, UnicodeError, json.JSONDecodeError, ValueError) as error:
            warnings.append(f"Skipped malformed file {path}: {error}")
            skipped_files += 1
            continue
        parsed_files += 1
        records_in_file = 0
        for index, record in enumerate(records):
            if isinstance(record, dict):
                task_id = record.get("task_id", record.get("id"))
                metadata = record.get("metadata", {})
                record_domain = record.get("domain", record.get("_source_domain",
                    metadata.get("domain") if isinstance(metadata, dict) else None))
            else:
                task_id = record_domain = None
            if task_ids and str(task_id) not in task_ids:
                continue
            if domain and str(record_domain or "").casefold() != domain.casefold():
                continue
            calls = extract_tool_calls(record)
            if not isinstance(record, (dict, list)):
                continue
            task_records += 1
            records_in_file += 1
            counts.update(calls)
            task_calls.append({"task_id": str(task_id or f"{path.name}#{index + 1}"), "calls": calls})
        if records_in_file:
            conversation_files += 1
        else:
            skipped_files += 1
            warnings.append(f"Skipped {path}: no supported conversation records matched.")

    report = {
        "input_paths": [str(path.resolve()) for path in input_paths],
        "discovered_files": [str(path) for path in sorted(files, key=lambda item: str(item).casefold())],
        "files_processed": len(files),
        "parsed_files": parsed_files,
        "skipped_files": skipped_files,
        "conversation_files_processed": conversation_files,
        "task_records_processed": task_records,
        "reference_tools": sorted(set(reference_tools)),
        "warnings": warnings,
        "task_calls": task_calls,
        **calculate_diversity(counts, reference_tools),
    }
    if report["unknown_tool_calls"]:
        report["warnings"].append("Unknown tool calls are reported but excluded from the score.")
    return report


def print_summary(report: dict[str, Any]) -> None:
    """Print a concise summary alongside the full JSON report."""
    score = report["normalized_tool_usage_diversity"]
    print(f"Tool Usage Diversity: {'undefined' if score is None else f'{score:.6f}'}")
    print(f"Files: {report['parsed_files']} parsed, {report['skipped_files']} skipped")
    print(f"Tasks: {report['task_records_processed']}; calls: {report['recognized_tool_calls']} recognized, "
          f"{report['unknown_tool_calls']} unknown")
    for tool, count in report["tool_call_counts"].items():
        print(f"  {tool}: {count} ({report['tool_call_proportions'][tool]:.3f})")
    for warning in report["warnings"]:
        print(f"Warning: {warning}")

## Deterministic validation

These small fixtures check the entropy calculation, extraction rules, unknown-tool policy, and a guaranteed single-tool perturbation. Run this cell before analyzing real data.

In [ ]:
TOOLS = ["alpha", "beta", "gamma", "delta"]

# Uniform, single-tool, partial, empty, unknown, and one-tool reference cases.
assert math.isclose(calculate_diversity({tool: 25 for tool in TOOLS}, TOOLS)["normalized_tool_usage_diversity"], 1.0)
assert calculate_diversity({"alpha": 100}, TOOLS)["normalized_tool_usage_diversity"] == 0.0
assert math.isclose(calculate_diversity({"alpha": 50, "beta": 50}, TOOLS)["normalized_tool_usage_diversity"], 0.5)
assert calculate_diversity({}, TOOLS)["normalized_tool_usage_diversity"] is None
unknown = calculate_diversity({"outside": 3}, TOOLS)
assert unknown["unknown_tool_call_counts"] == {"outside": 3}
assert unknown["normalized_tool_usage_diversity"] is None
assert calculate_diversity({"only": 3}, ["only"])["normalized_tool_usage_diversity"] == 0.0

# Text mentions and tool results are ignored; duplicate IDs count once.
fixture = {"messages": [
    {"role": "user", "content": "Call alpha"},
    {"role": "assistant", "content": "alpha mentioned", "tool_calls": [
        {"id": "1", "name": "alpha"}, {"id": "1", "name": "alpha"},
        {"id": "2", "function": {"name": "beta"}}]},
    {"role": "tool", "name": "gamma", "content": "result"},
    {"role": "assistant", "function_call": {"name": "delta"}},
]}
assert extract_tool_calls(fixture) == ["alpha", "beta", "delta"]

# A global one-tool perturbation must have zero diversity.
synthetic = {"task_calls": [{"calls": [tool]} for tool in TOOLS],
             "normalized_tool_usage_diversity": 1.0}
validation = validate_perturbations(synthetic, TOOLS)
assert validation["strict_single_tool"]["normalized_tool_usage_diversity"] == 0.0
assert validation["strict_single_tool"]["passed"]
assert validation["task_level_single_tool_subset"]["normalized_tool_usage_diversity"] > 0.0

# Exercise loading, extraction, scoring, and validation together on a JSON fixture.
import tempfile
with tempfile.TemporaryDirectory() as directory:
    fixture_path = Path(directory) / "tasks.json"
    fixture_path.write_text(json.dumps({"tasks": [
        {"task_id": tool, "messages": [{"role": "assistant", "tool_calls": [{"name": tool}]}]}
        for tool in TOOLS
    ]}), encoding="utf-8")
    dataset = analyze_dataset([fixture_path], TOOLS)
    end_to_end = validate_perturbations(dataset, TOOLS)
    assert dataset["recognized_tool_calls"] == 4
    assert math.isclose(dataset["normalized_tool_usage_diversity"], 1.0)
    assert end_to_end["strict_single_tool"]["normalized_tool_usage_diversity"] == 0.0
    assert math.copysign(1.0, end_to_end["strict_single_tool"]["normalized_tool_usage_diversity"]) == 1.0

print("All Tool Usage Diversity checks passed.")

## Analyze your data

Edit the paths and the tool inventory below before running this cell. The inventory must be complete and stay the same when comparing datasets. The final JSON report is written to the configured output path; the input dataset is not modified.

In [ ]:
# Replace these examples with your actual tau2 output paths and declared tools.
INPUT_PATHS = [Path(r"C:\path\to\tau2-results")]
REFERENCE_TOOLS = ["replace_with_tool_1", "replace_with_tool_2"]
OUTPUT_PATH = Path(r"C:\path\to\tool-diversity-report.json")

TASK_IDS = None  # Analyze tasks 0 and 1; set None to include all simulations.
DOMAIN = None  # Read from info.environment_info.domain_name in this results.json.

# Keep False until the paths and tool names above have been configured.
RUN_ANALYSIS = False #default is false
if RUN_ANALYSIS:
    report = analyze_dataset(INPUT_PATHS, REFERENCE_TOOLS, TASK_IDS, DOMAIN)
    report["validation"] = validate_perturbations(report, REFERENCE_TOOLS)
    report["configuration"] = {
        "reference_tools": sorted(set(REFERENCE_TOOLS)),
        "task_ids": sorted(TASK_IDS) if TASK_IDS else None,
        "domain": DOMAIN,
        "unknown_tool_policy": "reported and excluded from score",
    }
    print_summary(report)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(f"JSON report saved to {OUTPUT_PATH}")
else:
    print("Set RUN_ANALYSIS = True after updating the input paths and reference tools.")